<img height="200" src="https://raw.githubusercontent.com/mmattamala/LogosFCFM/master/Ingenier%C3%ADa%20El%C3%A9ctrica/fcfm_die_png.png">


# **Proyecto Final: Clasificación de Comandos de Voz con Arquitectura Transformer**
# EL4203-2 Programación Avanzada - Primavera 2026

---
**Integrantes del grupo**: Benjamín Guelfi, Joujin Ho Ku, Bianca Zamora

**Profesor**: Shaharyar Kamal

**Auxiliares**: Aaron Molina

**Ayudantes**: Sebastian Carrasco, Agustin Salgado, Nicolás Matamala y Felipe González

**Este proyecto debe ser desarrollado en grupos de máximo 3 estudiantes.**

# Descripción del proyecto

El **reconocimiento de palabras clave** (*Keyword Spotting* o KWS) es la tecnología central detrás de asistentes de voz y sistemas embebidos en tiempo real. Tradicionalmente este problema se abordaba con arquitecturas convolucionales (CNN), pero los avances recientes se fundamentan en **mecanismos de auto-atención y modelos Transformer** (*Attention Is All You Need*).

En este proyecto construirán un clasificador de comandos de voz para 12 clases sobre el dataset **Google Speech Commands v2** utilizando un **codificador Transformer puro** (sin capas convolucionales). Para asegurar un desarrollo progresivo y riguroso, el proyecto se divide en las siguientes etapas:

| Parte | Qué construye |
|---|---|
| 1 | Carga estructurada del dataset y particiones independientes | 
| 2 | Clase `SpeechCommandsDataset` y extracción de Log-Mel Spectrogram |
| 3 | Análisis acústico previo: Visualización y similitud espectral |
| 4 | Proyección de secuencias y Codificación Posicional | 
| 5 | Arquitectura del Codificador Transformer |
| 6 | Rutinas de optimización y ciclo de entrenamiento/validación | 
| 7 | Evaluación: Curvas de aprendizaje, Matriz de Confusión y Análisis Fonético |


# Logística y evaluación

**Entregables**
- Este cuaderno (`.ipynb`) completamente ejecutado, con todas las salidas, métricas y gráficos visibles.
- Un informe técnico breve en PDF (máximo 6 páginas) donde detallen la metodología, el impacto de los hiperparámetros explorados y el análisis de la matriz de confusión.

**Reglas y consideraciones**
- Se prohíbe el uso de capas convolucionales (`nn.Conv1d`, `nn.Conv2d`, etc.). El modelo debe aprender representaciones espectro-temporales usando exclusivamente proyecciones lineales y auto-atención.
- Cada parte incluye **tests públicos** usando `unittest` que validan contratos de entrada/salida, dimensiones y tipos de datos antes de avanzar a la siguiente sección.
- Para asegurar reproducibilidad en sus resultados, fijen siempre las semillas aleatorias (`torch.manual_seed` y `np.random.seed`).

# Conceptos necesarios

## 1. Extracción de características acústicas: Log-Mel Spectrogram
Un archivo de audio a 16 kHz es una secuencia temporal de 16.000 valores de presión por segundo. Para clasificar patrones vocálicos y consonánticos, transformar la señal al plano tiempo-frecuencia es indispensable.

El **Log-Mel Spectrogram** realiza este proceso en tres etapas:
1. Aplica la Transformada de Fourier de Tiempo Reducido (STFT) para descomponer la onda en ventanas temporales cortas.
2. Proyecta las frecuencias lineales a un banco de filtros triangulares espaciados según la **escala perceptual Mel** (`n_mels`), imitando la resolución no lineal del sistema auditivo humano.
3. Transforma las amplitudes a escala logarítmica de decibelios ($10 \log_{10}(P)$) mediante `AmplitudeToDB`.

El resultado es una matriz de dimensiones $(n\_mels, T)$, donde cada columna $t$ representa el espectro energético de la voz en ese instante temporal.

> **Observación sobre dimensionalidad:** El parámetro `n_mels` define cuántas bandas de frecuencia describen a cada paso de tiempo. En un enfoque secuencial temporal, valores moderados de `n_mels` generan modelos livianos, rápidos de entrenar y sumamente efectivos.

## 2. Cómo percibe el tiempo un Transformer: Proyección y Codificación Posicional
Dado que los bloques de auto-atención procesan todos los pasos temporales simultáneamente sin noción intrínseca de orden, debemos inyectar el tiempo explícitamente:
1. **Proyección lineal:** La columna espectral de tamaño `n_mels` en el instante $t$ se proyecta mediante una matriz de pesos a un espacio latente de dimensión $D$ (`dim_emb`).
2. **Codificación posicional (*Positional Encoding*):** A cada vector proyectado se le suma un vector determinista basado en funciones sinusoidales (Vaswani et al., 2017):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/D}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/D}}\right)$$

# Parte 1: Catálogo y particiones independientes por hablante (Entregado)

El siguiente bloque organiza el catálogo de metadatos para las 12 clases canónicas del benchmark:
- **10 palabras objetivo:** `yes`, `no`, `up`, `down`, `left`, `right`, `on`, `off`, `stop`, `go`.
- **Clase `_unknown_`:** Palabras secundarias que el sistema debe ignorar.
- **Clase `_silence_`:** Ruido de fondo y ausencia de comandos.

Ejecute esta celda para generar las particiones oficiales de entrenamiento, validación y prueba garantizando la separación estricta de hablantes.

In [30]:
import os
import numpy as np
import pandas as pd
import torch

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

clases = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go', '_unknown_', '_silence_']
clases_a_id = {c: i for i, c in enumerate(clases)}
id_a_clases = {i: c for i, c in enumerate(clases)}

def crear_catalogo_speech(n_por_clase=150):
    """Crea un catalogo con rutas, etiquetas y splits garantizando independencia de hablantes."""
    filas = []
    splits = ['train'] * int(n_por_clase * 0.7) + ['val'] * int(n_por_clase * 0.15) + ['test'] * int(n_por_clase * 0.15)
    for c_id, c_nom in enumerate(clases):
        for i, split in enumerate(splits):
            filas.append({
                'filepath': f"{c_nom}_{i:04d}.wav",
                'label': c_id,
                'clase': c_nom,
                'speaker_id': f"spk_{c_id}_{i % 25:03d}",
                'split': split
            })
    return pd.DataFrame(filas)

df_catalogo = crear_catalogo_speech(n_por_clase=150)
df_train = df_catalogo[df_catalogo['split'] == 'train'].reset_index(drop=True)
df_val = df_catalogo[df_catalogo['split'] == 'val'].reset_index(drop=True)
df_test = df_catalogo[df_catalogo['split'] == 'test'].reset_index(drop=True)

print(f"Catalogo estructurado:")
print(f" - Train: {len(df_train)} muestras ({len(df_train) // 12} por clase)")
print(f" - Val:   {len(df_val)} muestras")
print(f" - Test:  {len(df_test)} muestras")
df_train.head()

Catalogo estructurado:
 - Train: 1260 muestras (105 por clase)
 - Val:   264 muestras
 - Test:  264 muestras


,filepath,label,clase,speaker_id,split
0,yes_0000.wav,0,yes,spk_0_000,train
1,yes_0001.wav,0,yes,spk_0_001,train
2,yes_0002.wav,0,yes,spk_0_002,train
3,yes_0003.wav,0,yes,spk_0_003,train
4,yes_0004.wav,0,yes,spk_0_004,train


# Parte 2: Clase `SpeechCommandsDataset` (Log-Mel Spectrogram)

En esta etapa deben implementar la clase **`SpeechCommandsDataset`** heredando de `torch.utils.data.Dataset`.

**Requerimientos de implementación:**
1. `__init__(self, df, sample_rate=16000, n_fft=1024, hop_length=256, n_mels=40)`: recibe el DataFrame de metadatos y configura la transformación `torchaudio.transforms.MelSpectrogram` junto a `torchaudio.transforms.AmplitudeToDB`.
2. `__len__(self)`: retorna el total de muestras en el conjunto.
3. `__getitem__(self, idx)`:
   - Carga o prepara el audio asegurando que tenga exactamente `sample_rate` muestras (1 segundo). Si es más corto, aplica relleno con ceros (*zero-padding*); si es más largo, lo trunca.
   - Aplica el cálculo de Log-Mel Spectrogram (`MelSpectrogram` + `AmplitudeToDB`).
   - Retorna una tupla `(mel_spec_db, label)` donde `mel_spec_db` es un tensor 2D de dimensiones `(n_mels, T)` de tipo `torch.float32` y `label` es un escalar `torch.long`.

In [31]:
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T

class SpeechCommandsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, sample_rate: int = 16000, n_fft: int = 1024, hop_length: int = 256, n_mels: int = 40):
        self.df = df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        
        # 1. Se configura self.mel_transform
        self.mel_transform = T.MelSpectrogram(sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
        # 2. Se configura self.amplitude_to_db
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # 1. Obtener audio (o simular señal de 1 segundo a 16000 muestras)
        audio_path = self.df.iloc[idx]['filepath']   # Se obtiene la ruta del audio del indice requerido
        label = self.df.iloc[idx]['label']        # Se obtiene la etiqueta del audio del indice requerido 
        
        # 2. Ajustar longitud: zero-padding si len < self.sample_rate, truncar si len > self.sample_rate
        audio_sample, fs = torchaudio.load(audio_path)  # Se carga el audio y la frecuencia de muestreo

        if len(audio_sample[0]) < self.sample_rate: # Si la cantidad de muestras por canal es menor a la sample_rate
            # Zero-padding
            padding_size = self.sample_rate - len(audio_sample[0]) # Cuantas muestras faltan para completar 
            padding_zeros = torch.zeros((audio_sample.shape[0], padding_size)) # Se crea un tensor de ceros de muestras faltantes
            audio_sample = torch.cat((audio_sample, padding_zeros), dim=1) # Se concatenan las muestras originales y los ceros

        elif len(audio_sample[0]) >self.sample_rate: # Si la cantidad de muestras por canal es mayor a la sample_rate
            # Truncar
            audio_sample = audio_sample[:, :self.sample_rate]

        # 3. Calcular mel_spec =
        mel_spec = self.mel_transform(audio_sample)
        
        # 4. Calcular mel_spec_db =
        mel_spec_db = self.amplitude_to_db(mel_spec)

        # 5. Retornar (mel_spec_db, torch.tensor(label, dtype=torch.long))
        return (mel_spec_db, torch.tensor(label, dtype=torch.long))

# Parte 3: Análisis Acústico Previo — Visualización y Similitud Espectral

Antes de entrenar un modelo, inspeccionar las señales en el plano tiempo-frecuencia permite entender qué comandos comparten patrones acústicos.

**Tareas a desarrollar:**
1. **Visualización de espectrogramas:** Seleccione muestras de cuatro comandos (por ejemplo: `yes`, `no`, `stop` y `go`) y grafique sus Log-Mel Spectrograms en una cuadrícula $2 \times 2$ usando `plt.imshow` con mapa de color continuo (ej. `viridis` o `magma`). Añada etiquetas de frecuencia y tiempo.
2. **Matriz de correlación espectral promedio:**
   - Para cada una de las 12 clases, calcule el espectrograma promedio promediando sus muestras.
   - Aplane cada espectrograma promedio a un vector 1D y calcule la matriz de correlación de Pearson ($12 \times 12$) entre todas las palabras clave.
   - Grafique esta matriz de similitud como un mapa de calor (*heatmap*).
3. **Pregunta de análisis:** Observando la matriz de correlación acústica, ¿cuáles pares de palabras presentan la mayor correlación espectral? ¿Anticipa que el modelo tendrá problemas distinguiéndolas más adelante?

In [ ]:
import matplotlib.pyplot as plt

# Desarrolle aqui la visualizacion de espectrogramas y la matriz de correlacion promedio
print(df_train.head())
print(os.path.exists(df_train.iloc[0]['filepath']))

train_dataset = SpeechCommandsDataset(df_train)
val_dataset = SpeechCommandsDataset(df_val)
test_dataset = SpeechCommandsDataset(df_test)

       filepath  label clase speaker_id  split
0  yes_0000.wav      0   yes  spk_0_000  train
1  yes_0001.wav      0   yes  spk_0_001  train
2  yes_0002.wav      0   yes  spk_0_002  train
3  yes_0003.wav      0   yes  spk_0_003  train
4  yes_0004.wav      0   yes  spk_0_004  train
False


# Parte 4: Proyección Temporal y Codificación Posicional

El Transformer requiere que la secuencia de vectores temporales resida en un espacio latente de dimensión fija $D$ (`dim_emb`).

1. **`PositionalEncoding(dim_emb, max_len=300)`**: módulo de PyTorch (`nn.Module`) que precalcula la matriz estática de senos y cosenos y la suma a la secuencia en el método `forward(x)`.
2. **`ProyeccionTemporal(n_mels, dim_emb)`**: toma el tensor espectral de entrada de tamaño $(B, n\_mels, T)$, transpone los ejes a $(B, T, n\_mels)$ para que el tiempo sea la dimensión secuencial, proyecta cada instante con `nn.Linear(n_mels, dim_emb)` y suma la codificación posicional.

In [33]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, dim_emb: int, max_len: int = 300):
        super().__init__()

        # 1. Crear matriz PE de tamaño (1, max_len, dim_emb)
        # 2. Llenar columnas pares con sen(pos / 10000^(2i/dim_emb)) e impares con cos
        # 3. Registrar como buffer: self.register_buffer('pe', pe)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (Batch, Sequencia_T, dim_emb)
        # Sumar self.pe[:, :x.size(1), :] a x y retornar
        pass

class ProyeccionTemporal(nn.Module):
    def __init__(self, n_mels: int = 40, dim_emb: int = 64):
        super().__init__()
        # Inicializar nn.Linear(n_mels, dim_emb) y PositionalEncoding(dim_emb)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Entrada x: (Batch, n_mels, T)
        # Transponer a (Batch, T, n_mels), proyectar con nn.Linear y sumar posicion
        pass


# Parte 5: Arquitectura del Codificador Transformer

Implemente la clase **`AudioTransformerClassifier`** integrando los módulos desarrollados:

1. **Entrada:** `ProyeccionTemporal(n_mels, dim_emb)`.
2. **Codificador Transformer:** Bloque `nn.TransformerEncoderLayer(d_model=dim_emb, nhead=num_heads, dim_feedforward=dim_ff, batch_first=True)` apilado en $L$ capas mediante `nn.TransformerEncoder`.
3. **Agregación (*Temporal Pooling*):** Promedie la secuencia de tokens a lo largo del eje temporal (`x.mean(dim=1)`).
4. **Cabeza de clasificación:** Capa lineal `nn.Linear(dim_emb, num_classes)` para estimar los logits de las 12 clases.

In [34]:
class AudioTransformerClassifier(nn.Module):
    def __init__(self, n_mels: int = 40, num_classes: int = 12, dim_emb: int = 64, num_heads: int = 4, num_layers: int = 2, dim_ff: int = 128):
        super().__init__()
        # 1. self.proyeccion = ProyeccionTemporal(n_mels, dim_emb)
        # 2. nn.TransformerEncoderLayer con batch_first=True
        # 3. nn.TransformerEncoder con num_layers capas
        # 4. self.clasificador = nn.Linear(dim_emb, num_classes)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (Batch, n_mels, T)
        # 1. Proyectar temporalmente -> (Batch, T, dim_emb)
        # 2. Pasar por el encoder Transformer -> (Batch, T, dim_emb)
        # 3. Promediar secuencia en dim=1 -> (Batch, dim_emb)
        # 4. Proyectar con self.clasificador -> (Batch, num_classes)
        pass


# Parte 6: Rutinas de Optimización y Entrenamiento

Implemente la función:
```python
def entrenar_modelo(modelo, train_loader, val_loader, epocas: int, lr: float, device: str) -> dict:
```

**Requerimientos del contrato:**
- Debe instanciar la función de pérdida `nn.CrossEntropyLoss()` y el optimizador `torch.optim.Adam(modelo.parameters(), lr=lr)`.
- En cada época, debe alternar explícitamente entre el modo de entrenamiento (`modelo.train()`) y evaluación (`modelo.eval()`).
- En la fase de entrenamiento, debe iterar sobre los lotes, reiniciar gradientes, computar la pasada hacia adelante, retropropagar la pérdida y actualizar los parámetros.
- En la fase de validación, debe evaluar sin calcular gradientes (`torch.no_grad()`).
- Debe retornar un diccionario con el historial de métricas: `{'loss_train': [...], 'acc_train': [...], 'loss_val': [...], 'acc_val': [...]}` con valores flotantes por época.

In [35]:
def entrenar_modelo(modelo: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epocas: int = 6, lr: float = 1e-3, device: str = 'cpu') -> dict:
    # Implementar el ciclo completo siguiendo los requerimientos del contrato
    pass


# Parte 7: Evaluación, Exploración de Hiperparámetros y Matriz de Confusión

Esta sección final no incluye tests automáticos y se califica por la calidad técnica de sus experimentos, gráficos y análisis.

### 1. Guía de exploración de hiperparámetros recomendados
Se sugiere explorar el comportamiento del sistema variando sistemáticamente alguno de los siguientes rangos:
- **Número de filtros Mel (`n_mels`):** Explore valores en el rango **[12, 40]**. Observe cómo impacta en el tamaño del modelo y compruebe si valores moderados son suficientes para clasificar los comandos.
- **Dimensión de incrustación (`dim_emb`):** [32, 64, 128].
- **Capas del codificador (`num_layers`):** [1, 2, 4].
- **Tasa de aprendizaje (`lr`):** [5e-4, 1e-3, 3e-3].
- **Épocas:** 6 a 12 épocas son suficientes para observar convergencia.

### 2. Entregables de la evaluación:
1. **Curvas de aprendizaje:** Grafique la evolución de la pérdida (*loss*) y exactitud (*accuracy*) para entrenamiento y validación.
2. **Matriz de confusión:** Evalúe su mejor configuración sobre el conjunto de prueba (`df_test`), obtenga las predicciones y grafique la matriz de confusión normalizada (usando `matplotlib` o `seaborn`).
3. **Análisis de errores:**
   - Identifique los pares de comandos que presentan mayor confusión mutua (por ejemplo: *tree* vs. *three*, *go* vs. *no*, o confusiones con *_unknown_*).
   - **Pista (Hint):** *Analicen la transcripción fonética de las palabras en inglés y los sonidos. Conecten este análisis con los mapas de similitud espectral que generaron en la Parte 3 para explicar por qué el Transformer comete ciertos errores.*

In [36]:
# Desarrolle aqui el entrenamiento de su configuracion final,
# graficos de curvas de perdida y la visualizacion de la matriz de confusion
